In [1]:
import torch
from torch import nn
from torch.optim import Adam
from torch.utils.data import Dataset, DataLoader

In [2]:
# pip install pytorch-lightning
import pytorch_lightning as pl
from pytorch_lightning.loggers import CSVLogger, TensorBoardLogger
from pytorch_lightning.callbacks import (
    Callback, EarlyStopping, ModelCheckpoint, LearningRateMonitor,
    DeviceStatsMonitor, GradientAccumulationScheduler
)

# # pip install lightning
# import lightning.pytorch as pl
# from lightning.pytorch.loggers import CSVLogger, TensorBoardLogger
# from lightning.pytorch.callbacks import Callback, EarlyStopping, ModelCheckpoint

/Users/minhhuunguyen/REPOSITORY/my_env/lib/python3.9/site-packages/urllib3/__init__.py:34: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [3]:
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split

In [4]:
california_housing = fetch_california_housing()

In [5]:
X = california_housing.data
y = california_housing.target.reshape(-1, 1)

In [6]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [7]:
class CaliforniaHousingDataset(Dataset):
    def __init__(self, features, targets):
        self.features = torch.tensor(features, dtype=torch.float32)
        self.targets = torch.tensor(targets, dtype=torch.float32)

    def __len__(self):
        return len(self.features)

    def __getitem__(self, idx):
        return self.features[idx], self.targets[idx]

In [8]:
train_dataset = CaliforniaHousingDataset(X_train, y_train)

In [9]:
test_dataset = CaliforniaHousingDataset(X_test, y_test)

In [10]:
class CaliforniaHousingDataModule(pl.LightningDataModule):
    def __init__(self, train_dataset, test_dataset, batch_size=64):
        super().__init__()
        self.train_dataset = train_dataset
        self.test_dataset = test_dataset
        self.batch_size = batch_size

    def train_dataloader(self):
        return DataLoader(
            self.train_dataset,
            batch_size=self.batch_size,
            shuffle=True
        )

    def val_dataloader(self):
        return DataLoader(
            self.test_dataset,
            batch_size=self.batch_size
        )

In [11]:
data_module = CaliforniaHousingDataModule(train_dataset, test_dataset)

In [12]:
class SimpleNN(pl.LightningModule):
    def __init__(self):
        super(SimpleNN, self).__init__()
        self.fc1 = nn.Linear(8, 64)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(64, 1)

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        return x

    def training_step(self, batch, batch_idx):
        x, y = batch
        y_pred = self(x)
        loss = nn.MSELoss()(y_pred, y)
        self.log("train_loss", loss, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        x, y = batch
        y_pred = self(x)
        loss = nn.MSELoss()(y_pred, y)
        self.log("val_loss", loss, prog_bar=True)
        return loss

    def configure_optimizers(self):
        return Adam(self.parameters(), lr=0.001)

In [13]:
model = SimpleNN()
model

SimpleNN(
  (fc1): Linear(in_features=8, out_features=64, bias=True)
  (relu): ReLU()
  (fc2): Linear(in_features=64, out_features=1, bias=True)
)

In [14]:
experiment_name = 'TestPyTorchLightning'

In [15]:
class PrintCallback(Callback):
    def on_train_start(self, trainer, pl_module):
        print("Training is started!")
        print("Model summary...")
        print(pl_module)
        print("Number of training batches...", trainer.num_training_batches)
        print("Number of validation batches...", trainer.num_val_batches)

    def on_train_end(self, trainer, pl_module):
        print("Training is done.")

In [16]:
trainer = pl.Trainer(
    max_epochs=10,
    precision=32,
    callbacks=[
        PrintCallback(),
        EarlyStopping('val_loss'),
        ModelCheckpoint(dirpath='ckpts'),
        LearningRateMonitor(logging_interval='epoch'),
        DeviceStatsMonitor(cpu_stats=True),
        GradientAccumulationScheduler(scheduling={2: 2}) # from epoch 3, accumulate 2 batches
    ],
    logger=[
        CSVLogger(save_dir='csv_logs', name=experiment_name),
        TensorBoardLogger(save_dir='tb_logs', name=experiment_name)
    ],
    log_every_n_steps=1,
    check_val_every_n_epoch=1
)

GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs


In [17]:
# trainer = pl.Trainer(
#     accelerator="gpu",
#     devices=8,
#     strategy="ddp",
#     num_nodes=4
# )
# # https://lightning.ai/docs/pytorch/stable/accelerators/mps_basic.html

In [18]:
trainer.fit(model, data_module)

Missing logger folder: csv_logs/TestPyTorchLightning

  | Name | Type   | Params
--------------------------------
0 | fc1  | Linear | 576   
1 | relu | ReLU   | 0     
2 | fc2  | Linear | 65    
--------------------------------
641       Trainable params
0         Non-trainable params
641       Total params
0.003     Total estimated model params size (MB)


Sanity Checking: |                                                                                            …

Training is started!
Model summary...
SimpleNN(
  (fc1): Linear(in_features=8, out_features=64, bias=True)
  (relu): ReLU()
  (fc2): Linear(in_features=64, out_features=1, bias=True)
)
Number of training batches... 258
Number of validation batches... [65]


/Users/minhhuunguyen/REPOSITORY/my_env/lib/python3.9/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:441: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.
/Users/minhhuunguyen/REPOSITORY/my_env/lib/python3.9/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:441: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.


Training: |                                                                                                   …

...

Validation: |                                                                                                 …

Validation: |                                                                                                 …

Validation: |                                                                                                 …

`Trainer.fit` stopped: `max_epochs=10` reached.


Training is done.


In [19]:
%reload_ext tensorboard
%tensorboard --logdir=tb_logs/TestPyTorchLightning